# Ejercicio 12: Bases de Datos Vectoriales

## Leandro Bravo

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [47]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [48]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [49]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [50]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [51]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [52]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [53]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [54]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [55]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # similitud coseno con embeddings normalizados
print(f"Dimensión del índice: {dim}")
print(f"Tipo de índice: {type(index).__name__}")

Dimensión del índice: 768
Tipo de índice: IndexFlatIP


In [56]:
# 2. Cargar los embeddings al índice
index.add(embeddings.astype("float32"))
print(f"Índices añadidos: {index.ntotal}")

Índices añadidos: 79104


In [57]:
# 3. Realizar búsqueda a partir de una query
D, I = index.search(query_vec.astype("float32"), k=5)
print("Query:", query_text)
print("Distancias:", D)
print("Índices:", I)

for rank, idx in enumerate(I[0]):
    row = chunks_df.iloc[int(idx)]
    print(f"Rank: {rank+1} | Índice: {idx} | Similitud: {D[0, rank]:.4f}")
    print(f"Texto: {row['text']}\\n")

Query: Battery measuring
Distancias: [[0.8703487  0.86180055 0.8401017  0.83913374 0.8385891 ]]
Índices: [[10176     1 10177 37406 71872]]
Rank: 1 | Índice: 10176 | Similitud: 0.8703
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or its voltage output, to a more comprehensive testing of the battery's condition, namely its capacity for accumulating charge and any possible flaws affecting the battery's performance and security. The most simple battery tester is a DC ammeter, that indicates the battery's charge rate. DC voltmeters can be used to estimate the charge rate of a battery, provided that its nominal voltage is known. There are many types of integrated battery testers, each one corresponding to a specific condition testing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac\n
Rank: 2

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
for rank, idx in enumerate(I[0]):
    row = chunks_df.iloc[int(idx)]
    print(f"rank={rank+1} idx={idx} score={D[0, rank]:.4f}")
    print(row["text"])
    print() Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [58]:
from qdrant_client import QdrantClient, models
import numpy as np

client = QdrantClient(":memory:")
collection_name = "wiki_chunks"
dim = embeddings.shape[1]

client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=dim, distance=models.Distance.COSINE),
)

for i, row in chunks_df.iterrows():
    client.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=int(i),
                vector=embeddings[int(i)].astype("float32"),
                payload={
                    "text": row["text"],
                    "metadata": {
                        "doc_id": int(row["doc_id"]),
                        "chunk_id": int(row["chunk_id"]),
                    },
                },
            )
        ],
    )

print(f"Colección '{collection_name}' creada y cargada con {len(embeddings)} puntos")


def qdrant_search(query_embedding, k=5):
    q_vec = np.asarray(query_embedding).reshape(-1).astype("float32")
    results = client.query_points(
        collection_name=collection_name,
        query=q_vec,
        limit=k,
        with_payload=True,
    ).points
    return [
        (hit.id, hit.score, hit.payload.get("text"), hit.payload.get("metadata"))
        for hit in results
    ]

C:\Users\leand\AppData\Local\Temp\ipykernel_22492\4270744693.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Colección 'wiki_chunks' creada y cargada con 79104 puntos


In [59]:
# Ejemplo de consulta con k=5
results = qdrant_search(query_vec, k=5)

for item_id, score, text, metadata in results:
    print(f"id={item_id} | score={score:.4f} | text={text[:180]}")
    print(f"metadata={metadata}\n")

id=10176 | score=0.8703 | text=Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in
metadata={'doc_id': 1391, 'chunk_id': 0}

id=1 | score=0.8618 | text=Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's
metadata={'doc_id': 1, 'chunk_id': 0}

id=10177 | score=0.8401 | text=ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is based on the empirical fact that
metadata={'doc_id': 1391, 'chunk_id': 1}

id=37406 | score=0.8391 | text=ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the resistor under test. The indic
metadata={'doc_id': 5067, 'chunk_

## Respuestas

- La métrica usada fue cosine porque los embeddings fueron normalizados durante la generación, lo que hace que la similitud coseno sea apropiada y estable para comparar textos.
- Filtrar por metadata en Qdrant es mucho más sencillo que en FAISS, porque Qdrant soporta payloads y filtros nativos sobre campos de metadata.
- Al aumentar k, el tiempo de respuesta suele crecer ligeramente porque se recupera una lista más grande de resultados, aunque en conjuntos pequeños el cambio suele ser pequeño.

---

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [1]:
from pymilvus import MilvusClient, DataType
import numpy as np

# 1. Conectar a Milvus Lite (creará un archivo local llamado 'milvus_demo.db')
client = MilvusClient("milvus_demo.db")
dim = embeddings.shape[1] # 768

coll_exact = "wiki_exact"
coll_ann = "wiki_ann"

# Limpiar colecciones por si repites la ejecución de la celda
if client.has_collection(coll_exact): client.drop_collection(coll_exact)
if client.has_collection(coll_ann): client.drop_collection(coll_ann)

ConnectionConfigException: <ConnectionConfigException: (code=1, message=milvus-lite is required for local database connections. Please install it with: pip install pymilvus[milvus_lite])>

In [ ]:
# 2. Definir el Esquema
schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=dim)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65000)
schema.add_field(field_name="doc_id", datatype=DataType.INT64)

In [ ]:
# 3. Preparar Índices
# a) Índice FLAT para búsqueda exacta
index_exact = client.prepare_index_params()
index_exact.add_index(field_name="embedding", index_type="FLAT", metric_type="COSINE")

# b) Índice HNSW para búsqueda ANN (Approximate Nearest Neighbor)
index_ann = client.prepare_index_params()
index_ann.add_index(
    field_name="embedding", 
    index_type="HNSW", 
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 200} # Parámetros de construcción del grafo
)

# Crear las colecciones
client.create_collection(collection_name=coll_exact, schema=schema, index_params=index_exact)
client.create_collection(collection_name=coll_ann, schema=schema, index_params=index_ann)


In [ ]:
# 4. Insertar los Embeddings (en lotes para no saturar la memoria RAM)
print("Insertando datos... esto puede tardar un momento.")
batch_size = 5000
for i in range(0, len(chunks_df), batch_size):
    batch = chunks_df.iloc[i:i+batch_size]
    data_batch = []
    
    for j, row in batch.iterrows():
        data_batch.append({
            "id": int(j),
            "embedding": embeddings[j].tolist(),
            "text": row["text"][:5000], # Truncamos si es necesario para ajustar al VARCHAR
            "doc_id": int(row["doc_id"])
        })
    
    client.insert(collection_name=coll_exact, data=data_batch)
    client.insert(collection_name=coll_ann, data=data_batch)

print(f"¡Inserción completada! Total de registros: {len(chunks_df)}")

In [ ]:
import time

# 5. Función de búsqueda
def milvus_search(query_embedding, collection_name, k=5, search_params=None):
    # Milvus espera una lista de listas para los vectores de consulta
    q_vec = np.asarray(query_embedding).reshape(1, -1).astype("float32").tolist()
    
    start_time = time.time()
    results = client.search(
        collection_name=collection_name,
        data=q_vec,
        limit=k,
        search_params=search_params,
        output_fields=["text", "doc_id"]
    )
    end_time = time.time()
    
    latency = end_time - start_time
    
    parsed_results = []
    for hits in results:
        for hit in hits:
            parsed_results.append({
                "id": hit["id"], 
                "score": hit["distance"], 
                "text": hit["entity"]["text"], 
                "doc_id": hit["entity"]["doc_id"]
            })
            
    return parsed_results, latency




In [ ]:
# 6. Ejecución
res_exact_5, t_exact_5 = milvus_search(query_vec, coll_exact, k=5)
# Para HNSW, usamos 'ef' en los parámetros de búsqueda. Un 'ef' bajo sacrifica precisión por velocidad.
res_ann_5, t_ann_5 = milvus_search(query_vec, coll_ann, k=5, search_params={"metric_type": "COSINE", "params": {"ef": 10}})

ids_exact_5 = set([r["id"] for r in res_exact_5])
ids_ann_5 = set([r["id"] for r in res_ann_5])
overlap_5 = len(ids_exact_5.intersection(ids_ann_5))

print(f"Búsqueda Exacta (FLAT) - Tiempo: {t_exact_5:.4f}s")
print(f"Búsqueda ANN (HNSW)    - Tiempo: {t_ann_5:.4f}s")
print(f"Overlap (coincidencias): {overlap_5} de 5\n")


print(" k=20 ")
res_exact_20, t_exact_20 = milvus_search(query_vec, coll_exact, k=20)
# Ajustamos 'ef' a 20 (justo al límite de k) para ver si la aproximación pierde algún documento del top real
res_ann_20, t_ann_20 = milvus_search(query_vec, coll_ann, k=20, search_params={"metric_type": "COSINE", "params": {"ef": 20}})
ids_exact_20 = set([r["id"] for r in res_exact_20])
ids_ann_20 = set([r["id"] for r in res_ann_20])
overlap_20 = len(ids_exact_20.intersection(ids_ann_20))

print(f"Búsqueda Exacta (FLAT) - Tiempo: {t_exact_20:.4f}s")
print(f"Búsqueda ANN (HNSW)    - Tiempo: {t_ann_20:.4f}s")
print(f"Overlap (coincidencias): {overlap_20} de 20\n")


## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [ ]:
import weaviate
import weaviate.classes.config as wc
import weaviate.classes.query as wq
import numpy as np

# 1. Conectar a Weaviate 
# (Usamos EmbeddedOptions que levanta un Weaviate temporal en memoria/disco local en Linux/WSL)
from weaviate.embedded import EmbeddedOptions
client = weaviate.WeaviateClient(
    embedded_options=EmbeddedOptions()
)
client.connect()

# Limpiar colección si ya existe para evitar errores al re-ejecutar
if client.collections.exists("WikiChunk"):
    client.collections.delete("WikiChunk")

In [ ]:
# 2. Definir el Esquema (Clase y Propiedades)
wiki_collection = client.collections.create(
    name="WikiChunk",
    properties=[
        wc.Property(name="text", data_type=wc.DataType.TEXT),
        wc.Property(name="doc_id", data_type=wc.DataType.INT),
        wc.Property(name="chunk_id", data_type=wc.DataType.INT),
    ],
    # Configuramos para usar nuestros propios vectores (no pedir a Weaviate que los calcule)
    vectorizer_config=wc.Configure.Vectorizer.none(),
)

In [ ]:
# 3. Insertar objetos con propiedades + vector
print("Insertando datos en Weaviate... (Usaremos los primeros 2000 para que sea rápido)")
with wiki_collection.batch.dynamic() as batch:
    for i, row in chunks_df.head(2000).iterrows():
        batch.add_object(
            properties={
                "text": str(row["text"]),
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"])
            },
            vector=embeddings[i].tolist() # Pasamos nuestro vector pre-calculado de E5
        )
print("¡Inserción completa!")

In [ ]:
# 4. Función de búsqueda (Top-k) con filtro opcional
def weaviate_search(query_embedding, k=5, filter_doc_id=None):
    # Aseguramos que el vector sea una lista plana de floats
    q_vec = np.asarray(query_embedding).reshape(-1).tolist()
    
    # 5. Lógica opcional de filtrado por metadata
    filters = None
    if filter_doc_id is not None:
        filters = wq.Filter.by_property("doc_id").equal(filter_doc_id)
        
    # Ejecutar búsqueda near_vector
    response = wiki_collection.query.near_vector(
        near_vector=q_vec,
        limit=k,
        return_metadata=wq.MetadataQuery(distance=True),
        filters=filters
    )
    
    # Formatear la salida para cumplir con el entregable
    results = []
    for obj in response.objects:
        results.append({
            "id": obj.uuid,
            "score": obj.metadata.distance,
            "text": obj.properties["text"],
            "doc_id": obj.properties["doc_id"]
        })
    return results

In [ ]:
# Probamos la búsqueda Normal
print("--- Búsqueda Top-k Normal ---")
res_normal = weaviate_search(query_vec, k=3)
for r in res_normal:
    print(f"Doc_ID: {r['doc_id']} | Distancia: {r['score']:.4f}")
    print(f"Texto: {r['text'][:120]}...\n")

# Probamos la búsqueda con Filtro (Solo queremos resultados del doc_id 1391, si existiera en los primeros 2000)
print("--- Búsqueda Top-k con Filtro (doc_id = 1) ---")
res_filtrada = weaviate_search(query_vec, k=3, filter_doc_id=1)
for r in res_filtrada:
    print(f"Doc_ID: {r['doc_id']} | Distancia: {r['score']:.4f}")
    print(f"Texto: {r['text'][:120]}...\n")

# Finalmente, cerramos la conexión
client.close()

## Respuestas
* La diferencia radica en que el modelo "tabla + filas" de SQL es muy rígido y bidimensional; el vector suele ser tratado simplemente como una columna más. En cambio, "schema + objetos" (como en Weaviate) tiene una naturaleza más parecida a un grafo estructurado (o NoSQL documental). Cada objeto tiene una identidad (UUID), y el vector no es un dato adjunto, sino que es la representación semántica central del objeto mismo.

* Al configurar Weaviate, la complejidad inicial es mayor. Tuvimos que definir explícitamente clases, tipos de datos (DataType.TEXT, DataType.INT) y configurar el comportamiento del motor (como apagar el vectorizador automático). Sin embargo, a cambio ganamos alta expresividad: podemos hacer consultas híbridas, aplicar validación estricta de tipos al insertar datos, y ejecutar filtros complejos por propiedades semánticas (Filter.by_property), algo que en soluciones más simples o de bajo nivel (como FAISS) requeriría construir lógica condicional completamente desde cero por nuestra cuenta.

---

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [ ]:
import chromadb
import numpy as np

# 1. Crear el cliente de Chroma (modo en memoria para prototipado rápido)
chroma_client = chromadb.Client()

# Limpiamos la colección por si re-ejecutas la celda
try:
    chroma_client.delete_collection(name="wiki_collection")
except:
    pass

In [ ]:
# 2. Crear la colección
collection = chroma_client.create_collection(
    name="wiki_collection",
    metadata={"hnsw:space": "cosine"} # Le indicamos explícitamente que use similitud coseno
)

In [ ]:
# 3. Preparar los datos
# IMPORTANTE: Chroma exige que los IDs sean strings, no enteros.
ids = [str(i) for i in chunks_df.index]
docs = chunks_df["text"].tolist()
metadatas = [{"doc_id": int(row["doc_id"]), "chunk_id": int(row["chunk_id"])} for _, row in chunks_df.iterrows()]
embs = embeddings.tolist()

In [ ]:
# 4. Insertar en lotes
print("Insertando datos en Chroma... (al no definir esquema, esto es directo pero toma unos segundos)")
batch_size = 5000
for i in range(0, len(ids), batch_size):
    collection.add(
        ids=ids[i:i+batch_size],
        embeddings=embs[i:i+batch_size],
        documents=docs[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size]
    )
    
print(f"¡Inserción completa de {collection.count()} documentos!")

In [ ]:
# 5. Función de búsqueda
def chroma_search(query_embedding, k=5):
    # Chroma espera una lista plana para el vector de búsqueda
    q_vec = np.asarray(query_embedding).reshape(-1).tolist()
    
    # Ejecutamos el query
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Formatear la salida desanidando el diccionario de Chroma
    parsed_results = []
    # Usamos [0] porque consultamos un solo vector; Chroma soporta multi-query
    for i in range(len(results['ids'][0])):
        parsed_results.append({
            "id": results['ids'][0][i],
            "score": results['distances'][0][i], # En cosine, menor distancia = mayor similitud en Chroma
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i]
        })
    return parsed_results

In [ ]:
# 6. Prueba de búsqueda Top-k
print("--- Resultados de Búsqueda en Chroma (k=5) ---")
res_chroma = chroma_search(query_vec, k=5)

for r in res_chroma:
    print(f"ID: {r['id']} | Distancia: {r['score']:.4f}")
    print(f"Metadata: {r['metadata']}")
    print(f"Texto: {r['text'][:150]}...\n")

## Respuestas

* Fue inmensamente más fácil y directo. Chroma se siente como usar un diccionario nativo de Python con superpoderes. No hubo necesidad de definir esquemas estrictos (DDL), tipos de datos (VARCHAR, INT64), ni preocuparse por configurar los hiperparámetros de los algoritmos de indexación (HNSW, IVF). Todo entra y sale como listas planas.

* Al ser una herramienta diseñada primordialmente para prototipado rápido e interactuar con frameworks de LLMs, Chroma (en su versión local) sufriría en escenarios de altísima concurrencia (miles de queries por segundo). Además, herramientas como Milvus o Qdrant ofrecen arquitecturas distribuidas robustas (para escalar horizontalmente en la nube), control de acceso basado en roles (RBAC) y gestión de memoria más eficiente a escala masiva, características que son vitales en un entorno de producción real y crítico.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?
